# Warm Start GP: 热启动遗传编程增强 Alpha 挖掘

## 论文参考

- **标题**: Alpha Mining and Enhancing via Warm Start Genetic Programming
- **作者**: Weizhe Ren
- **年份**: 2024

### 摘要

本文提出了一种热启动遗传编程 (Warm Start GP) 方法，通过用已知优秀 Alpha 因子 (动量、均值回复、量价因子)
初始化 GP 种群，加速进化收敛并提升最终因子质量。
实验表明热启动 GP 在 IC/IR 指标上显著优于随机初始化的冷启动 GP。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 热启动 vs 冷启动

**冷启动 (Cold Start)**: 种群完全随机初始化，从零开始搜索

**热启动 (Warm Start)**: 种群中注入已知有效的 Alpha 因子种子：

**1. 种子因子库**

- **动量因子**: $\alpha = \text{close} - \text{delay}(\text{close}, d)$
- **均值回复因子**: $\alpha = \text{ts\_mean}(\text{close}, d) - \text{close}$
- **量价因子**: $\alpha = \text{rank}(\text{volume}) - \text{rank}(\text{close})$
- **波动率因子**: $\alpha = \text{ts\_std}(\text{close}, d)$

**2. 种群初始化策略**

$$P_0 = \underbrace{\{s_1, s_2, \ldots, s_k\}}_{\text{种子因子}} \cup \underbrace{\{r_1, r_2, \ldots, r_{N-k}\}}_{\text{随机个体}}$$

种子因子占种群的 30%，其余 70% 仍然随机初始化以保持多样性。

**3. 优势**

- 更快收敛: 初始种群已包含有效模式
- 更高上限: 种子因子通过交叉/变异产生更强的组合因子
- 保持多样性: 随机个体避免早熟收敛

In [ ]:
# ============================================================
# Cell 3: 动画展示 - Warm Start vs Cold Start 收敛对比
# ============================================================
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)
n_gen = 30

# 模拟冷启动收敛 (从低IC开始, 慢速收敛)
cold_best = [0.005]
cold_avg = [0.001]
for g in range(1, n_gen):
    imp = np.random.exponential(0.002) * (1 - 0.5 * g / n_gen)
    cold_best.append(min(cold_best[-1] + imp, 0.06))
    cold_avg.append(cold_best[-1] * np.random.uniform(0.2, 0.5))

# 模拟热启动收敛 (从较高IC开始, 快速收敛到更高值)
warm_best = [0.025]  # 种子因子提供较好起点
warm_avg = [0.015]
for g in range(1, n_gen):
    imp = np.random.exponential(0.003) * (1 - 0.4 * g / n_gen)
    warm_best.append(min(warm_best[-1] + imp, 0.085))
    warm_avg.append(warm_best[-1] * np.random.uniform(0.4, 0.7))

gens = list(range(n_gen))

frames = []
for step in range(1, n_gen + 1):
    frames.append(go.Frame(
        data=[
            go.Scatter(x=gens[:step], y=warm_best[:step], mode='lines+markers',
                       name='Warm Start (Best IC)', line=dict(color='#F44336', width=3),
                       marker=dict(size=4)),
            go.Scatter(x=gens[:step], y=warm_avg[:step], mode='lines',
                       name='Warm Start (Avg IC)', line=dict(color='#F44336', width=1.5, dash='dash')),
            go.Scatter(x=gens[:step], y=cold_best[:step], mode='lines+markers',
                       name='Cold Start (Best IC)', line=dict(color='#2196F3', width=3),
                       marker=dict(size=4)),
            go.Scatter(x=gens[:step], y=cold_avg[:step], mode='lines',
                       name='Cold Start (Avg IC)', line=dict(color='#2196F3', width=1.5, dash='dash')),
        ],
        name=f'Gen {step-1}'
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text='Warm Start vs Cold Start: IC 收敛对比'),
        xaxis_title='代数 (Generation)', yaxis_title='IC (绝对值)',
        yaxis=dict(range=[0, 0.1]),
        height=500, width=900,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 200}, 'transition': {'duration': 100}}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
        )],
    )
)
fig.show()

In [ ]:
# ============================================================
# Cell 4: 环境设置与导入
# ============================================================
import sys
import os
import copy
import random
import warnings
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

sys.path.insert(0, '..')
from shared.data_fetcher import get_etf_daily, get_index_daily
from shared.backtest_engine import Backtester
from shared.visualization import (
    plot_equity_curve, plot_drawdown, plot_metrics_table,
    plot_cumulative_comparison
)
from shared.constants import DEFAULT_START, DEFAULT_END, ETF_CSI300

print('所有模块导入成功')

In [ ]:
# ============================================================
# Cell 5: 数据获取
# ============================================================
try:
    df = get_etf_daily(ETF_CSI300, DEFAULT_START, DEFAULT_END)
    print(f'ETF 510300 数据: {len(df)} 条')
except Exception as e:
    print(f'数据获取异常: {e}, 使用模拟数据')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    n = len(dates)
    np.random.seed(42)
    price = 4.5 + np.cumsum(np.random.randn(n) * 0.02)
    price = np.maximum(price, 2.0)
    df = pd.DataFrame({
        'open': price * (1 + np.random.randn(n) * 0.003),
        'close': price,
        'high': price * (1 + np.abs(np.random.randn(n) * 0.008)),
        'low': price * (1 - np.abs(np.random.randn(n) * 0.008)),
        'volume': np.random.randint(50000, 2000000, n).astype(float),
    }, index=dates)

forward_ret = df['close'].pct_change(5).shift(-5)
print(f'数据范围: {df.index[0].strftime("%Y-%m-%d")} ~ {df.index[-1].strftime("%Y-%m-%d")}')

In [ ]:
# ============================================================
# Cell 6: GP 引擎 + 种子因子库
# ============================================================

PRIMITIVES = ['close', 'open', 'high', 'low', 'volume']
UNARY_OPS = ['rank']
BINARY_OPS = ['add', 'sub', 'mul', 'div']
TS_OPS = ['ts_mean', 'ts_std', 'delay']
TS_WINDOWS = [3, 5, 10, 20]


class Node:
    def __init__(self, op=None, children=None, value=None, window=None):
        self.op = op
        self.children = children or []
        self.value = value
        self.window = window

    def is_leaf(self):
        return self.op is None

    def depth(self):
        if self.is_leaf(): return 1
        return 1 + max(c.depth() for c in self.children)

    def __str__(self):
        if self.is_leaf(): return self.value
        if self.op in UNARY_OPS: return f'{self.op}({self.children[0]})'
        if self.op in BINARY_OPS:
            sym = {'add':'+','sub':'-','mul':'*','div':'/'}[self.op]
            return f'({self.children[0]} {sym} {self.children[1]})'
        if self.op in TS_OPS: return f'{self.op}({self.children[0]}, {self.window})'
        return '?'

    def evaluate(self, data):
        if self.is_leaf(): return data[self.value].astype(float).copy()
        if self.op == 'rank': return self.children[0].evaluate(data).rank(pct=True)
        if self.op == 'add': return self.children[0].evaluate(data) + self.children[1].evaluate(data)
        if self.op == 'sub': return self.children[0].evaluate(data) - self.children[1].evaluate(data)
        if self.op == 'mul': return self.children[0].evaluate(data) * self.children[1].evaluate(data)
        if self.op == 'div':
            b = self.children[1].evaluate(data).replace(0, np.nan)
            return self.children[0].evaluate(data) / b
        if self.op == 'ts_mean': return self.children[0].evaluate(data).rolling(self.window, min_periods=1).mean()
        if self.op == 'ts_std': return self.children[0].evaluate(data).rolling(self.window, min_periods=2).std().fillna(0)
        if self.op == 'delay': return self.children[0].evaluate(data).shift(self.window).fillna(method='bfill')
        return pd.Series(0, index=data.index)


def random_tree(max_depth=5, depth=0):
    if depth >= max_depth or (depth > 0 and random.random() < 0.3):
        return Node(value=random.choice(PRIMITIVES))
    r = random.random()
    if r < 0.15:
        return Node(op=random.choice(UNARY_OPS), children=[random_tree(max_depth, depth+1)])
    elif r < 0.55:
        return Node(op=random.choice(BINARY_OPS),
                    children=[random_tree(max_depth, depth+1), random_tree(max_depth, depth+1)])
    else:
        return Node(op=random.choice(TS_OPS),
                    children=[random_tree(max_depth, depth+1)],
                    window=random.choice(TS_WINDOWS))


def get_all_nodes(tree):
    nodes = [tree]
    for c in tree.children: nodes.extend(get_all_nodes(c))
    return nodes

def crossover(t1, t2):
    t1, t2 = copy.deepcopy(t1), copy.deepcopy(t2)
    n1, n2 = get_all_nodes(t1), get_all_nodes(t2)
    if len(n1) < 2 or len(n2) < 2: return t1, t2
    a, b = random.choice(n1[1:]), random.choice(n2[1:])
    a.op, b.op = b.op, a.op
    a.children, b.children = b.children, a.children
    a.value, b.value = b.value, a.value
    a.window, b.window = b.window, a.window
    if t1.depth() > 6: t1 = random_tree(5)
    if t2.depth() > 6: t2 = random_tree(5)
    return t1, t2

def mutate(tree, max_depth=5):
    t = copy.deepcopy(tree)
    nodes = get_all_nodes(t)
    target = random.choice(nodes)
    sub = random_tree(max_depth=3)
    target.op, target.children, target.value, target.window = sub.op, sub.children, sub.value, sub.window
    return t if t.depth() <= 6 else random_tree(max_depth)

def compute_ic(alpha_values, fwd_ret):
    valid = pd.DataFrame({'a': alpha_values, 'r': fwd_ret}).dropna()
    if len(valid) < 30: return 0.0
    ic, _ = stats.spearmanr(valid['a'], valid['r'])
    return ic if not np.isnan(ic) else 0.0

def evaluate_fitness(tree, data, fwd_ret):
    try:
        alpha = tree.evaluate(data)
        if alpha.isna().all() or alpha.std() == 0: return -1.0
        return abs(compute_ic(alpha, fwd_ret))
    except: return -1.0


# ---- 种子因子库 (已知优秀Alpha) ----
def make_seed_factors():
    """构建已知有效的Alpha因子表达式树"""
    seeds = []
    # 1. 动量因子: close - delay(close, d)
    for d in [5, 10, 20]:
        tree = Node(op='sub', children=[
            Node(value='close'),
            Node(op='delay', children=[Node(value='close')], window=d)
        ])
        seeds.append(tree)

    # 2. 均值回复: ts_mean(close, d) - close
    for d in [10, 20]:
        tree = Node(op='sub', children=[
            Node(op='ts_mean', children=[Node(value='close')], window=d),
            Node(value='close'),
        ])
        seeds.append(tree)

    # 3. 量价背离: rank(volume) - rank(close)
    tree = Node(op='sub', children=[
        Node(op='rank', children=[Node(value='volume')]),
        Node(op='rank', children=[Node(value='close')]),
    ])
    seeds.append(tree)

    # 4. 波动率: ts_std(close, d)
    for d in [10, 20]:
        tree = Node(op='ts_std', children=[Node(value='close')], window=d)
        seeds.append(tree)

    # 5. 高低价差: high - low
    tree = Node(op='sub', children=[Node(value='high'), Node(value='low')])
    seeds.append(tree)

    # 6. 量价动量: rank(volume * (close - open))
    tree = Node(op='rank', children=[
        Node(op='mul', children=[
            Node(value='volume'),
            Node(op='sub', children=[Node(value='close'), Node(value='open')])
        ])
    ])
    seeds.append(tree)

    return seeds

seed_factors = make_seed_factors()
print(f'种子因子数量: {len(seed_factors)}')
for i, sf in enumerate(seed_factors):
    ic = evaluate_fitness(sf, df, forward_ret)
    print(f'  Seed {i}: IC={ic:.4f} | {sf}')

In [ ]:
# ============================================================
# Cell 7: 对比实验 - Warm Start vs Cold Start GP
# ============================================================

POP_SIZE = 50
N_GEN = 30
MAX_DEPTH = 5
TOURNAMENT = 5
CX_PROB = 0.7
MUT_PROB = 0.2
ELITE = 5

def run_gp(data, fwd_ret, init_population, label='GP'):
    """运行GP进化并返回历史记录"""
    pop = [copy.deepcopy(ind) for ind in init_population]
    history = {'gen': [], 'best_ic': [], 'avg_ic': []}

    for gen in range(N_GEN):
        fits = [evaluate_fitness(ind, data, fwd_ret) for ind in pop]
        valid = [f for f in fits if f >= 0]
        best_fit = max(fits)
        avg_fit = np.mean(valid) if valid else 0

        history['gen'].append(gen)
        history['best_ic'].append(best_fit)
        history['avg_ic'].append(avg_fit)

        if gen % 10 == 0:
            best_idx = np.argmax(fits)
            print(f'  [{label}] Gen {gen}: Best IC={best_fit:.4f}, Avg IC={avg_fit:.4f}')

        # 进化
        sorted_idx = np.argsort(fits)[::-1]
        elites = [copy.deepcopy(pop[i]) for i in sorted_idx[:ELITE]]
        new_pop = list(elites)

        while len(new_pop) < POP_SIZE:
            c1 = random.sample(range(POP_SIZE), TOURNAMENT)
            c2 = random.sample(range(POP_SIZE), TOURNAMENT)
            p1 = copy.deepcopy(pop[max(c1, key=lambda i: fits[i])])
            p2 = copy.deepcopy(pop[max(c2, key=lambda i: fits[i])])
            if random.random() < CX_PROB: p1, p2 = crossover(p1, p2)
            if random.random() < MUT_PROB: p1 = mutate(p1, MAX_DEPTH)
            if random.random() < MUT_PROB: p2 = mutate(p2, MAX_DEPTH)
            new_pop.extend([p1, p2])
        pop = new_pop[:POP_SIZE]

    # 返回最佳个体
    final_fits = [evaluate_fitness(ind, data, fwd_ret) for ind in pop]
    best_idx = np.argmax(final_fits)
    return pop[best_idx], final_fits[best_idx], history


# --- Cold Start: 纯随机初始化 ---
print('=== Cold Start GP ===')
random.seed(42); np.random.seed(42)
cold_pop = [random_tree(MAX_DEPTH) for _ in range(POP_SIZE)]
cold_best, cold_ic, cold_hist = run_gp(df, forward_ret, cold_pop, 'Cold')

# --- Warm Start: 30% 种子 + 70% 随机 ---
print('\n=== Warm Start GP ===')
random.seed(42); np.random.seed(42)
n_seeds = min(len(seed_factors), int(POP_SIZE * 0.3))
warm_pop = [copy.deepcopy(sf) for sf in seed_factors[:n_seeds]]
# 种子因子的变异体
while len(warm_pop) < int(POP_SIZE * 0.3):
    warm_pop.append(mutate(random.choice(seed_factors), MAX_DEPTH))
# 补充随机个体
while len(warm_pop) < POP_SIZE:
    warm_pop.append(random_tree(MAX_DEPTH))
warm_best, warm_ic, warm_hist = run_gp(df, forward_ret, warm_pop, 'Warm')

print(f'\n=== 结果对比 ===')
print(f'Cold Start: IC={cold_ic:.4f} | {cold_best}')
print(f'Warm Start: IC={warm_ic:.4f} | {warm_best}')
print(f'IC 提升: {(warm_ic - cold_ic) / max(cold_ic, 0.001):.1%}')

In [ ]:
# ============================================================
# Cell 8: 信号生成与回测 (两种策略对比)
# ============================================================

def alpha_to_signals(tree, data, threshold=0.5):
    """将Alpha因子转为交易信号"""
    alpha = tree.evaluate(data).dropna()
    alpha_z = (alpha - alpha.rolling(60).mean()) / alpha.rolling(60).std()
    alpha_z = alpha_z.dropna()
    signals = pd.Series(0, index=alpha_z.index)
    signals[alpha_z > threshold] = 1
    signals[alpha_z < -threshold] = -1
    return signals

# 获取基准
try:
    bench = get_index_daily('sh000300', DEFAULT_START, DEFAULT_END)
    bench_prices = bench['close']
except:
    bench_prices = None

backtester = Backtester(initial_capital=1_000_000, t_plus_1=True)

# Cold Start 策略
cold_signals = alpha_to_signals(cold_best, df)
cold_result = backtester.run(df['close'], cold_signals, bench_prices)

# Warm Start 策略
warm_signals = alpha_to_signals(warm_best, df)
warm_result = backtester.run(df['close'], warm_signals, bench_prices)

print('=== Cold Start 回测结果 ===')
for k, v in cold_result['metrics'].items():
    print(f'  {k}: {v}')

print('\n=== Warm Start 回测结果 ===')
for k, v in warm_result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 绩效可视化
# ============================================================
import matplotlib.pyplot as plt

# 1. 收敛曲线对比
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(warm_hist['gen'], warm_hist['best_ic'], 'r-o', markersize=3, label='Warm Start (Best)')
ax1.plot(cold_hist['gen'], cold_hist['best_ic'], 'b-o', markersize=3, label='Cold Start (Best)')
ax1.fill_between(warm_hist['gen'], warm_hist['avg_ic'], alpha=0.2, color='red', label='Warm Avg')
ax1.fill_between(cold_hist['gen'], cold_hist['avg_ic'], alpha=0.2, color='blue', label='Cold Avg')
ax1.set_xlabel('代数'); ax1.set_ylabel('IC')
ax1.set_title('GP 进化收敛对比'); ax1.legend(); ax1.grid(True, alpha=0.3)

# 2. 策略累计收益对比
strategies = {
    'Warm Start GP': warm_result['returns'],
    'Cold Start GP': cold_result['returns'],
}
plot_cumulative_comparison(strategies, title='Warm Start vs Cold Start - 策略对比')

# 3. 绩效对比表
comparison = {}
for k in warm_result['metrics']:
    comparison[k] = f"Warm: {warm_result['metrics'][k]}  |  Cold: {cold_result['metrics'][k]}"
plot_metrics_table(comparison, title='Warm vs Cold 绩效对比')

# 4. Warm Start 回撤
plot_drawdown(warm_result['equity_curve'], title='Warm Start GP - 回撤')

## 结果讨论

### 策略表现

1. **收敛速度**: Warm Start 在早期代数即达到较高 IC，Cold Start 需要更多代数追赶
2. **最终质量**: Warm Start 的最终最佳 IC 通常高于 Cold Start
3. **多样性保持**: 70% 随机个体保证了种群不会过早收敛到种子因子附近

### 局限性

- 种子因子的选择依赖领域知识，可能引入偏差
- 小种群和少代数下结果随机性较大
- 单资产回测简化了实际应用场景

### 改进方向

- 动态调整种子比例: 随进化进行逐步减少种子影响
- 多目标优化: 同时优化 IC、ICIR 和因子换手率
- 因子库自动扩充: 将每次进化的优胜者加入种子库